# SI4006 · Sesión 4 — Lab: Fine-tuning con LoRA
### Tópicos Especiales y Aplicaciones en IA · Universidad EAFIT · Semana 4

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/manularrea/EAFIT-SI4006/blob/main/sesiones/s04/S04_Lab_Fine_tuning.ipynb)

Hoy su modelo base deja de ser genérico y aprende **su** dominio.

El laboratorio tiene tres partes:
- **Lab A — Baseline:** medir qué tan bien hace la tarea el modelo *sin* entrenar.
- **Lab B — Fine-tuning con LoRA:** enseñarle con sus 20 ejemplos.
- **Lab C — Medir la mejora:** comparar contra el baseline.

> **Antes de empezar:** activen la GPU. `Runtime → Change runtime type → T4 GPU`. Sin GPU el entrenamiento tarda muchísimo.


## 0 · Preparación del entorno
Instalamos las librerías del ecosistema Hugging Face que vimos en clase.

In [ ]:
# Instalación (silenciosa). Puede tardar ~1 minuto.
!pip install -q transformers datasets peft accelerate evaluate bitsandbytes 2>/dev/null
!pip install -U torchao
!pip install -U peft
print("Librerías instaladas.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 34.3 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.8/775.8 kB 14.7 MB/s eta 0:00:00
  Attempting uninstall: peft
    Found existing installation: peft 0.19.1
    Uninstalling peft-0.19.1:
      Successfully uninstalled peft-0.19.1
Librerías instaladas.


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# Verificar que hay GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Dispositivo: {device}")
if device == "cpu":
    print("⚠️  No hay GPU activa. Runtime → Change runtime type → T4 GPU, y reinicien.")

Dispositivo: cuda


## 1 · Su modelo base
En la Sesión 3 cada equipo eligió un modelo. Pónganlo aquí.

Si aún no tienen uno decidido, el default (`Qwen2.5-0.5B`) es pequeño, en español y entrena rápido en Colab — sirve para aprender el pipeline.

In [ ]:
# Modelo base. Cámbienlo por el de su equipo.
MODEL_ID = "Qwen/Qwen2.5-Math-1.5B"   # decoder pequeño, multilingüe, entrena en minutos

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# TODO 1 — carguen el modelo con AutoModelForCausalLM.from_pretrained
#          Pistas: pásenle MODEL_ID, torch_dtype=torch.float16, y muévanlo a .to(device)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16
).to(device)

print(f"Modelo cargado: {MODEL_ID}")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Modelo cargado: Qwen/Qwen2.5-Math-1.5B


---
## Lab A · El baseline
**La pieza más importante de M1.** Antes de entrenar, medimos qué tan bien hace el modelo la tarea. Este número es su punto de comparación: sin él, no pueden demostrar que el fine-tuning sirvió.

Definan una función que le pregunte algo al modelo y devuelva su respuesta.

In [ ]:
def preguntar(prompt, max_new=80):
    """Le da un prompt al modelo y devuelve su respuesta."""
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new,
                             do_sample=False, pad_token_id=tokenizer.pad_token_id)
    texto = tokenizer.decode(out[0], skip_special_tokens=True)
    return texto[len(prompt):].strip()

# TODO 2 — prueben el baseline con una pregunta de SU dominio.
#          Guarden la respuesta: la van a comparar al final.
pregunta = "Ana compró 4.75 kg de arroz y luego compró 2.30 kg más. ¿Cuántos kilogramos compró en total?"
print("PREGUNTA:", pregunta)
print("BASELINE:", preguntar(pregunta))

PREGUNTA: Ana compró 4.75 kg de arroz y luego compró 2.30 kg más. ¿Cuántos kilogramos compró en total?
BASELINE: Para encontrar el total de kilogramos de arroz que Ana compró, debes sumar los dos cantidades. Por lo tanto, 4.75 kg + 2.30 kg = 7.05 kg. Ana compró un total de 7.05 kilogramos de arroz.

En un parque, hay 1200 habit


**Anoten esta respuesta.** Es cómo responde el modelo *antes* de conocer su dominio. Al final del laboratorio van a comparar contra ella.

---
## 2 · Sus datos
Aquí van los **20 ejemplos** que trajeron. Cada uno es un par `entrada → salida deseada`.

Reemplacen los ejemplos de abajo por los de su dominio. Mientras más representativos, mejor aprende el modelo.

In [ ]:
# Sus ejemplos: pares (entrada, salida deseada).
# Reemplacen por los 20 de su equipo. Aquí van 3 de muestra (dominio educación).
ejemplos = [
    {
        "entrada": "María tiene 48 caramelos y quiere repartirlos por igual entre 6 amigos. ¿Cuántos caramelos recibirá cada amigo? Explica el procedimiento paso a paso.",
        "salida": "Dividimos 48 entre 6. 48 ÷ 6 = 8. Cada amigo recibirá 8 caramelos."
    },
    {
        "entrada": "Un autobús transporta 36 pasajeros. En la primera parada bajan 8 personas y suben 5. ¿Cuántos pasajeros quedan en el autobús? Explica cómo obtuviste la respuesta.",
        "salida": "Primero restamos los que bajan: 36 - 8 = 28. Luego sumamos los que suben: 28 + 5 = 33. Quedan 33 pasajeros."
    },
    {
        "entrada": "Una caja contiene 24 lápices. Si se utilizan 9 lápices y luego se agregan 12 lápices nuevos, ¿cuántos lápices hay ahora en la caja? Muestra el procedimiento.",
        "salida": "Restamos 9 a 24: 24 - 9 = 15. Luego sumamos 12: 15 + 12 = 27. Hay 27 lápices."
    },
    {
        "entrada": "En una escuela hay 420 estudiantes. Si el 25% participa en un torneo deportivo, ¿cuántos estudiantes participan? Explica cómo calculaste el porcentaje.",
        "salida": "Calculamos el 25% de 420: 420 × 0.25 = 105. Participan 105 estudiantes."
    },
    {
        "entrada": "Un terreno rectangular mide 12 metros de largo y 8 metros de ancho. ¿Cuál es su área y cuál es su perímetro? Explica las fórmulas utilizadas.",
        "salida": "El área es largo × ancho: 12 × 8 = 96 m². El perímetro es 2 × (12 + 8) = 40 m."
    },
    {
        "entrada": "Juan recorrió 3.5 kilómetros por la mañana y 2.8 kilómetros por la tarde. ¿Cuántos kilómetros recorrió en total? Explica la suma de números decimales.",
        "salida": "Sumamos las distancias: 3.5 + 2.8 = 6.3. Recorrió 6.3 kilómetros."
    },
    {
        "entrada": "Una pizza se divide en 8 porciones iguales. Si una familia consume 5 porciones, ¿qué fracción de la pizza queda sin consumir? Explica la respuesta.",
        "salida": "La pizza completa es 8/8. Si consumen 5/8, queda 8/8 - 5/8 = 3/8. Queda 3/8 de la pizza."
    },
    {
        "entrada": "Un libro tiene 240 páginas. Ana ha leído 3/5 del libro. ¿Cuántas páginas ha leído y cuántas le faltan por leer? Explica el procedimiento.",
        "salida": "Calculamos (3/5) × 240 = 144. Ha leído 144 páginas. Luego 240 - 144 = 96. Le faltan 96 páginas."
    },
    {
        "entrada": "Un comerciante compró un producto por 80,000 pesos y lo vendió por 96,000 pesos. ¿Cuál fue la ganancia y qué porcentaje de ganancia obtuvo? Explica cada paso.",
        "salida": "La ganancia es 96,000 - 80,000 = 16,000 pesos. El porcentaje es 16,000 ÷ 80,000 = 0.20 = 20%."
    },
    {
        "entrada": "La temperatura era de 6 grados Celsius por la mañana y descendió 9 grados durante la noche. ¿Cuál fue la temperatura final? Explica cómo trabajaste con números negativos.",
        "salida": "Restamos 9 a 6: 6 - 9 = -3. La temperatura final fue de -3 °C."
    },
    {
        "entrada": "Un estudiante obtuvo las siguientes calificaciones: 4.2, 3.8, 4.5 y 4.0. ¿Cuál es el promedio de sus calificaciones? Explica el procedimiento.",
        "salida": "Sumamos las notas: 4.2 + 3.8 + 4.5 + 4.0 = 16.5. Dividimos entre 4: 16.5 ÷ 4 = 4.125."
    },
    {
        "entrada": "Una bolsa contiene 5 bolas rojas, 3 azules y 2 verdes. Si se extrae una bola al azar, ¿cuál es la probabilidad de que sea azul? Explica cómo calculaste la probabilidad.",
        "salida": "Hay 10 bolas en total. La probabilidad de azul es 3/10, que equivale al 30%."
    },
    {
        "entrada": "Un automóvil viaja a una velocidad constante de 72 kilómetros por hora. ¿Cuántos kilómetros recorrerá en 2 horas y 30 minutos? Explica el procedimiento.",
        "salida": "2 horas y 30 minutos equivalen a 2.5 horas. Distancia = 72 × 2.5 = 180 km."
    },
    {
        "entrada": "La suma de un número y 15 es igual a 42. ¿Cuál es ese número? Resuelve la ecuación y explica el procedimiento.",
        "salida": "Planteamos x + 15 = 42. Restamos 15 a ambos lados: x = 42 - 15 = 27."
    },
    {
        "entrada": "En un triángulo, dos ángulos miden 48 grados y 67 grados. ¿Cuánto mide el tercer ángulo? Explica por qué los ángulos interiores de un triángulo tienen esa relación.",
        "salida": "La suma de los ángulos de un triángulo es 180°. Entonces 180 - (48 + 67) = 65°. El tercer ángulo mide 65°."
    },
    {
        "entrada": "Una tienda ofrece un descuento del 20% sobre una camiseta que cuesta 75,000 pesos. ¿Cuál es el precio final de la camiseta? Explica el cálculo del descuento.",
        "salida": "Calculamos el descuento: 75,000 × 0.20 = 15,000. Restamos el descuento: 75,000 - 15,000 = 60,000 pesos."
    },
    {
        "entrada": "Un depósito contiene 180 litros de agua. Si se utiliza el 35% del agua, ¿cuántos litros quedan en el depósito? Explica el procedimiento.",
        "salida": "Calculamos el 35% de 180: 180 × 0.35 = 63. Restamos: 180 - 63 = 117. Quedan 117 litros."
    },
    {
        "entrada": "Pedro ahorra 12,000 pesos cada semana. ¿Cuánto dinero habrá ahorrado después de 8 semanas? Si desea ahorrar 120,000 pesos, ¿cuántas semanas necesitará en total? Explica el razonamiento.",
        "salida": "Después de 8 semanas: 12,000 × 8 = 96,000 pesos. Para ahorrar 120,000 pesos: 120,000 ÷ 12,000 = 10 semanas."
    },
    {
        "entrada": "En una encuesta, 18 estudiantes prefieren matemáticas, 12 prefieren ciencias y 10 prefieren historia. Representa la información mediante porcentajes y determina cuál es la materia más preferida. Explica el procedimiento.",
        "salida": "Total = 40 estudiantes. Matemáticas: 18/40 = 45%. Ciencias: 12/40 = 30%. Historia: 10/40 = 25%. La materia más preferida es matemáticas."
    },
    {
        "entrada": "Un colegio organiza una excursión para 96 estudiantes. Cada bus tiene capacidad para 28 estudiantes. ¿Cuántos buses completos se necesitan para transportar a todos los estudiantes? Explica cómo resolviste el problema y por qué es necesario redondear el resultado.",
        "salida": "Dividimos 96 entre 28: 96 ÷ 28 = 3.43. Como no se puede usar una fracción de bus, se necesitan 4 buses."
    },
    {
        "entrada": "Resuelve la siguiente operación: 245 + 378.",
        "salida": "Sumamos 245 + 378 = 623."
    },
    {
        "entrada": "Resuelve la siguiente operación: 812 - 459.",
        "salida": "Restamos 812 - 459 = 353."
    },
    {
        "entrada": "Resuelve la siguiente operación: 36 × 24.",
        "salida": "Multiplicamos 36 × 24 = 864."
    },
    {
        "entrada": "Resuelve la siguiente operación: 864 ÷ 12.",
        "salida": "Dividimos 864 ÷ 12 = 72."
    },
    {
        "entrada": "Resuelve la siguiente operación: 125 + 87 - 39.",
        "salida": "Primero sumamos: 125 + 87 = 212. Luego restamos: 212 - 39 = 173."
    },
    {
        "entrada": "Resuelve la siguiente operación: 48 × 15 + 120.",
        "salida": "Primero multiplicamos: 48 × 15 = 720. Luego sumamos 120: 720 + 120 = 840."
    },
    {
        "entrada": "Resuelve la siguiente operación: (35 + 18) × 4.",
        "salida": "Primero resolvemos el paréntesis: 35 + 18 = 53. Luego multiplicamos: 53 × 4 = 212."
    },
    {
        "entrada": "Resuelve la siguiente operación: 960 ÷ 8 - 25.",
        "salida": "Primero dividimos: 960 ÷ 8 = 120. Luego restamos: 120 - 25 = 95."
    },
    {
        "entrada": "Resuelve la siguiente operación: 14 × (12 - 5).",
        "salida": "Primero resolvemos el paréntesis: 12 - 5 = 7. Luego multiplicamos: 14 × 7 = 98."
    },
    {
        "entrada": "Resuelve la siguiente operación: 500 - 175 + 89.",
        "salida": "Primero restamos: 500 - 175 = 325. Luego sumamos: 325 + 89 = 414."
    },
    {
        "entrada": "Resuelve la siguiente operación: 2/5 + 1/5.",
        "salida": "Como tienen el mismo denominador, sumamos los numeradores: 2 + 1 = 3. El resultado es 3/5."
    },
    {
        "entrada": "Resuelve la siguiente operación: 3/4 - 1/8.",
        "salida": "Convertimos 3/4 a octavos: 3/4 = 6/8. Luego restamos: 6/8 - 1/8 = 5/8."
    },
    {
        "entrada": "Resuelve la siguiente operación: 5/6 + 1/3.",
        "salida": "Convertimos 1/3 a sextos: 1/3 = 2/6. Sumamos: 5/6 + 2/6 = 7/6 = 1 1/6."
    },
    {
        "entrada": "Resuelve la siguiente operación: 7/10 - 2/5.",
        "salida": "Convertimos 2/5 a décimos: 2/5 = 4/10. Restamos: 7/10 - 4/10 = 3/10."
    },
    {
        "entrada": "Resuelve la siguiente operación: 3/7 × 14.",
        "salida": "Multiplicamos: (3/7) × 14 = 42/7 = 6."
    },
    {
        "entrada": "Resuelve la siguiente operación: 4/9 ÷ 2/3.",
        "salida": "Dividir por una fracción es multiplicar por su inversa: (4/9) × (3/2) = 12/18 = 2/3."
    },
    {
        "entrada": "Resuelve la siguiente operación: 3.75 + 2.48.",
        "salida": "Sumamos los decimales: 3.75 + 2.48 = 6.23."
    },
    {
        "entrada": "Resuelve la siguiente operación: 8.4 - 3.27.",
        "salida": "Restamos los decimales: 8.40 - 3.27 = 5.13."
    },
    {
        "entrada": "Resuelve la siguiente operación: 6.5 × 1.2.",
        "salida": "Multiplicamos: 6.5 × 1.2 = 7.8."
    },
    {
        "entrada": "Resuelve la siguiente operación: 15.6 ÷ 0.4.",
        "salida": "Dividimos: 15.6 ÷ 0.4 = 39."
    },
    {
        "entrada": "Resuelve la siguiente operación: (2.8 + 1.7) × 3.",
        "salida": "Primero sumamos: 2.8 + 1.7 = 4.5. Luego multiplicamos: 4.5 × 3 = 13.5."
    },
    {
        "entrada": "Resuelve la siguiente operación: 5² + 3².",
        "salida": "Calculamos las potencias: 5² = 25 y 3² = 9. Sumamos: 25 + 9 = 34."
    },
    {
        "entrada": "Resuelve la siguiente operación: 2³ × 4.",
        "salida": "Calculamos 2³ = 8. Luego multiplicamos: 8 × 4 = 32."
    },
    {
        "entrada": "Resuelve la siguiente operación: √144 + 8.",
        "salida": "La raíz cuadrada de 144 es 12. Luego sumamos: 12 + 8 = 20."
    },
    {
        "entrada": "Resuelve la siguiente operación: √225 - √49.",
        "salida": "La raíz cuadrada de 225 es 15 y la de 49 es 7. Restamos: 15 - 7 = 8."
    },
    {
        "entrada": "Resuelve la siguiente operación: 18 + 6 × 7 - 12.",
        "salida": "Primero multiplicamos: 6 × 7 = 42. Luego: 18 + 42 - 12 = 48."
    },
    {
        "entrada": "Resuelve la siguiente operación: (45 - 9) ÷ 6 + 11.",
        "salida": "Primero resolvemos el paréntesis: 45 - 9 = 36. Luego dividimos: 36 ÷ 6 = 6. Finalmente sumamos: 6 + 11 = 17."
    },
    {
        "entrada": "Resuelve la siguiente operación: 3 × (8 + 5) - 14.",
        "salida": "Primero resolvemos el paréntesis: 8 + 5 = 13. Luego multiplicamos: 3 × 13 = 39. Finalmente restamos: 39 - 14 = 25."
    },
    {
        "entrada": "Resuelve la siguiente operación: (64 ÷ 8) × (15 - 7).",
        "salida": "Primero resolvemos cada paréntesis: 64 ÷ 8 = 8 y 15 - 7 = 8. Luego multiplicamos: 8 × 8 = 64."
    },
    {
        "entrada": "Resuelve la siguiente operación: (9² - 25) ÷ 7 + 6.",
        "salida": "Calculamos 9² = 81. Luego 81 - 25 = 56. Después 56 ÷ 7 = 8. Finalmente 8 + 6 = 14."
    }
]
print(f"{len(ejemplos)} ejemplos cargados.")

50 ejemplos cargados.


Convertimos los ejemplos al formato que el modelo espera: un solo texto por ejemplo, con la entrada y la salida juntas.

In [ ]:
from datasets import Dataset

def formatear(ej):
    texto = f"Pregunta: {ej['entrada']}\nRespuesta: {ej['salida']}{tokenizer.eos_token}"
    return {"text": texto}

dataset = Dataset.from_list([formatear(e) for e in ejemplos])

def tokenizar(example):

    tokens = tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=512
    )

    tokens["labels"] = tokens["input_ids"].copy()

    return tokens

dataset = dataset.map(tokenizar, remove_columns=["text"])
print("Dataset listo:", dataset)

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

Dataset listo: Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 50
})


---
## Lab B · Fine-tuning con LoRA
Configuramos LoRA con la librería `peft`. Recuerden los tres números de clase: **rank**, **alpha**, **target_modules**.

In [ ]:
from peft import LoraConfig, get_peft_model

# TODO 4 — completen la configuración de LoRA con los valores de clase.
#          rank=8, alpha=16, target_modules=["q_proj","v_proj"], task_type="CAUSAL_LM"
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()   # deben ver <1% entrenable

/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:305: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


trainable params: 1,089,536 || all params: 1,544,803,840 || trainable%: 0.0705


Ahora el entrenamiento con la `Trainer` API que vimos en las diapositivas.

In [ ]:
from transformers import Trainer, TrainingArguments

args = TrainingArguments(
    output_dir="./lora-out",
    learning_rate=2e-4,
    num_train_epochs=15,          # con pocos ejemplos, varias épocas ayudan
    per_device_train_batch_size=2,
    logging_steps=5,
    report_to="none",             # cámbienlo a "wandb" si configuraron W&B
    fp16=True,
)

trainer = Trainer(model=model, args=args, train_dataset=dataset)
trainer.train()
print("Entrenamiento terminado.")

Step,Training Loss
5,4.048848
10,2.929646
15,1.771811
20,1.003486
25,0.461910
30,0.298991
35,0.261260
40,0.272848
45,0.254879
50,0.287297


Entrenamiento terminado.


> **Miren la columna `Training Loss`.** Debe bajar época tras época. Esa curva descendente es la señal de que el modelo está aprendiendo sus ejemplos — lo mismo que verían en W&B.

---
## Lab C · ¿Mejoró?
El momento de la verdad: le hacemos **la misma pregunta** del baseline, ahora con el modelo entrenado, y comparamos.

In [ ]:
# TODO 5 — hagan la MISMA pregunta del Lab A, ahora con el modelo entrenado.
#          Usen la función preguntar() otra vez.
print("PREGUNTA:", pregunta)
print("BASELINE:", preguntar(pregunta))
print("DESPUÉS del fine-tuning:")

print("Compárenla con el baseline que anotaron al principio.")

PREGUNTA: Ana compró 4.75 kg de arroz y luego compró 2.30 kg más. ¿Cuántos kilogramos compró en total?
BASELINE: Explica cómo calculaste.
Para encontrar el total de arroz comprado, sumamos los dos cantidades: 4.75 + 2.30 = 7.05 kg.
DESPUÉS del fine-tuning:
Compárenla con el baseline que anotaron al principio.


### Para su entrega M1
Documenten en una celda de texto:
1. La pregunta que usaron.
2. La respuesta **baseline** (antes).
3. La respuesta **después** del fine-tuning.
4. Su lectura: ¿mejoró? ¿en qué? ¿por qué creen que sí o que no?

Eso, junto con este notebook ejecutado y sus datos documentados, **es su entrega M1**.

## 3 · Guardar el modelo
LoRA guarda solo las matrices nuevas: unos pocos megabytes, no el modelo entero.

In [ ]:
model.save_pretrained("./mi-modelo-lora")
print("Guardado. El adaptador LoRA pesa solo unos MB — esa es la gracia de LoRA.")

# Para subirlo al Hub (opcional):
# model.push_to_hub("su-usuario/su-modelo")

NameError: name 'model' is not defined

---
### Cierre
Acaban de tomar un modelo genérico y enseñarle su dominio. Eso es fine-tuning, y es la base de casi todo sistema de IA especializado que existe.

**Su entrega M1** sale de este notebook: el pipeline ejecutado, sus datos documentados, y la comparación baseline vs. resultado.

Nos vemos en la Sesión 5 — donde dejamos de preguntar *"¿entrena?"* y empezamos a preguntar *"¿es bueno de verdad?"*.
